# Benchmark Results EDA

### Table of contents

1. [Data loading](#data-loading)
2. [Overview](#overview)
3. [F1 performance](#f1-performance)
   - [Overall accuracy](#overall-accuracy)
   - [Exact vs. fuzzy gap](#exact-vs-fuzzy-gap)
   - [Per-field breakdown](#per-field-breakdown)
4. [Score distributions](#score-distributions)
5. [Efficiency](#efficiency)
   - [Cost vs. accuracy](#cost-vs-accuracy)
   - [Latency](#latency)
6. [Conclusions](#conclusions)

## Introduction

This notebook explores the benchmark results across 12 experiments — three providers (Claude, Gemini, GPT), two model tiers (lite, standard), and two extraction strategies (agentic multimodal, single-pass text). Each experiment was run on 83 NDA documents from the Kleister-NDA dev split.


In [117]:
import json
import math
from pathlib import Path

import altair as alt
import polars as pl

## Data loading


In [118]:
data_path = Path.cwd() / "data" / "results"

df_aggregated = pl.read_csv(data_path / "benchmark-aggregated.csv")
df_per_document = pl.read_csv(data_path / "benchmark-per-document.csv")

In [119]:
lambert = 81.77
human = 97.86

## Benchmark

In [127]:
df_aggregated.select(
    pl.col(
        "provider",
        "tier",
        "strategy",
        "exact_f1_mean",
        "exact_f1_std",
        "latency_mean",
        "latency_p95",
        "cost_mean",
        "cost_std",
    )
).sort("exact_f1_mean", descending=True).head(5)

provider,tier,strategy,exact_f1_mean,exact_f1_std,latency_mean,latency_p95,cost_mean,cost_std
str,str,str,f64,f64,f64,f64,f64,f64
"""gemini""","""standard""","""agentic""",0.918359,0.125763,14.567474,31.293001,0.010635,0.006469
"""gemini""","""standard""","""single_pass""",0.915347,0.125566,9.789317,30.069521,0.00748,0.006002
"""gpt""","""standard""","""single_pass""",0.898883,0.154057,2.068752,2.992403,0.010944,0.004249
"""gemini""","""lite""","""single_pass""",0.89567,0.151544,1.413686,1.946596,0.001075,0.000449
"""gpt""","""standard""","""agentic""",0.895367,0.165382,8.452288,10.846992,0.024124,0.005973


In [121]:
def make_provider_facet_plot(df: pl.DataFrame, metric: str) -> alt.Chart:
    bars = (
        alt.Chart()
        .mark_bar()
        .encode(
            x="strategy:N",
            y=alt.Y(f"mean({metric}):Q").title(f"Mean {metric}"),
            color="strategy:N",
            tooltip=[f"mean({metric}):Q"],
        )
    )

    error_bars = (
        alt.Chart()
        .mark_errorbar(extent="ci")
        .encode(x="strategy:N", y=alt.Y(f"mean({metric}):Q").title(f"Mean {metric}"))
    )

    return (
        alt.layer(bars, error_bars, data=df)
        .properties(width=150, height=200)
        .facet(column="provider:N", row="tier:N")
    )


make_provider_facet_plot(df_aggregated, "exact_f1_mean")

alt.FacetChart(...)

In [122]:
records = df_aggregated.select(
    pl.col("provider", "tier", "strategy", "exact_f1_mean", "latency_mean", "cost_mean")
).to_dicts()

latency_max = max(r["latency_mean"] for r in records)
cost_max = max(r["cost_mean"] for r in records)


def make_tab_data(records, metric):
    return [
        {
            "tier": r["tier"],
            "provider": r["provider"],
            "strategy": r["strategy"],
            "value": r[metric],
        }
        for r in records
    ]


benchmark = {
    "series": [
        {"key": "agentic", "label": "Agentic", "color": "primary"},
        {"key": "single_pass", "label": "Single Pass", "color": "secondary"},
    ],
    "tabs": [
        {
            "id": "f1",
            "label": "F1 Score",
            "chartType": "faceted-bar",
            "unit": "%",
            "yDomain": [0, 1],
            "facet": {"rowKey": "tier", "colKey": "provider"},
            "xKey": "strategy",
            "yKey": "value",
            "data": make_tab_data(records, "exact_f1_mean"),
        },
        {
            "id": "latency",
            "label": "Latency",
            "chartType": "faceted-bar",
            "unit": "s",
            "yDomain": [0, math.ceil(latency_max)],
            "facet": {"rowKey": "tier", "colKey": "provider"},
            "xKey": "strategy",
            "yKey": "value",
            "data": make_tab_data(records, "latency_mean"),
        },
        {
            "id": "cost",
            "label": "Cost",
            "chartType": "faceted-bar",
            "unit": "$",
            "yDomain": [0, round(math.ceil(cost_max * 1000) / 1000, 3)],
            "facet": {"rowKey": "tier", "colKey": "provider"},
            "xKey": "strategy",
            "yKey": "value",
            "data": make_tab_data(records, "cost_mean"),
        },
    ],
}

with open(data_path / "blog" / "benchmark.json", "w") as f:
    json.dump(benchmark, f, indent=2)

## Per-field breakdown


In [123]:
def plot_field_heatmap(df: pl.DataFrame) -> alt.Chart:
    field_cols = [
        "exact_effective_date_f1_mean",
        "exact_party_f1_mean",
        "exact_jurisdiction_f1_mean",
        "exact_term_f1_mean",
    ]
    melted = (
        df.with_columns(
            pl.concat_str(
                [
                    pl.col("provider"),
                    pl.lit(" "),
                    pl.col("tier"),
                    pl.lit(" "),
                    pl.col("strategy"),
                ]
            ).alias("experiment")
        )
        .select(["experiment"] + field_cols)
        .unpivot(index="experiment", variable_name="column", value_name="f1")
        .with_columns(
            pl.col("column")
            .str.replace("exact_", "")
            .str.replace("_f1_mean", "")
            .alias("field")
        )
    )
    return (
        alt.Chart(melted.to_pandas())
        .mark_rect()
        .encode(
            x=alt.X(
                "field:N",
                title=None,
                sort=["effective_date", "party", "jurisdiction", "term"],
            ),
            y=alt.Y("experiment:N", title=None),
            color=alt.Color(
                "f1:Q",
                title="Exact F1",
                scale=alt.Scale(scheme="blues", domain=[0.5, 1.0]),
            ),
            tooltip=[
                alt.Tooltip("experiment:N"),
                alt.Tooltip("field:N"),
                alt.Tooltip("f1:Q", format=".3f", title="Exact F1"),
            ],
        )
        .properties(title="Per-field Exact F1", width=280, height=300)
    )


plot_field_heatmap(df_aggregated)

alt.Chart(...)

## Efficiency

### Cost vs. accuracy


In [125]:
def plot_cost_vs_f1(df: pl.DataFrame) -> alt.Chart:
    df_pd = df.with_columns(
        pl.concat_str(
            [
                pl.col("provider"),
                pl.lit(" "),
                pl.col("tier"),
                pl.lit(" "),
                pl.col("strategy"),
            ]
        ).alias("label")
    ).to_pandas()
    return (
        alt.Chart(df_pd)
        .mark_point(filled=True, size=100)
        .encode(
            x=alt.X(
                "cost_mean:Q",
                title="Mean cost per document (USD)",
                axis=alt.Axis(format="$.4f"),
            ),
            y=alt.Y(
                "exact_f1_mean:Q",
                title="Exact F1",
                scale=alt.Scale(domain=[0.6, 1.0]),
            ),
            color=alt.Color("provider:N", title="Provider"),
            shape=alt.Shape("tier:N", title="Tier"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("exact_f1_mean:Q", format=".3f", title="Exact F1"),
                alt.Tooltip("cost_mean:Q", format="$.5f", title="Cost/doc"),
                alt.Tooltip("cost_total:Q", format="$.3f", title="Total cost"),
            ],
        )
        .properties(title="Cost vs. Accuracy Trade-off", width=380, height=280)
    )


plot_cost_vs_f1(df_aggregated)

alt.Chart(...)

### Latency


In [126]:
def plot_latency(df: pl.DataFrame) -> alt.LayerChart:
    df_pd = df.with_columns(
        pl.concat_str(
            [
                pl.col("provider"),
                pl.lit(" "),
                pl.col("tier"),
                pl.lit(" "),
                pl.col("strategy"),
            ]
        ).alias("label")
    ).to_pandas()
    bars = (
        alt.Chart(df_pd)
        .mark_bar()
        .encode(
            x=alt.X("latency_mean:Q", title="Latency (s)"),
            y=alt.Y("label:N", sort="-x", title=None),
            color=alt.Color("provider:N", title="Provider"),
            tooltip=[
                alt.Tooltip("label:N", title="Experiment"),
                alt.Tooltip("latency_mean:Q", format=".2f", title="Mean (s)"),
                alt.Tooltip("latency_median:Q", format=".2f", title="Median (s)"),
                alt.Tooltip("latency_p95:Q", format=".2f", title="P95 (s)"),
            ],
        )
    )
    p95 = (
        alt.Chart(df_pd)
        .mark_tick(color="black", thickness=2, size=15)
        .encode(
            x=alt.X("latency_p95:Q"),
            y=alt.Y("label:N", sort="-x"),
            tooltip=[alt.Tooltip("latency_p95:Q", format=".2f", title="P95 (s)")],
        )
    )
    return (bars + p95).properties(
        title="Mean Latency per Document (tick = P95)", width=380, height=320
    )


plot_latency(df_aggregated)

alt.LayerChart(...)

## Conclusions

- **Gemini dominates at standard tier**, achieving the highest exact F1 (~0.918) across both strategies, with negligible difference between agentic-multimodal and single-pass text. This suggests its vision capabilities are not strictly necessary for this task at standard scale.

- **Single-pass text is the best value at lite tier**. Gemini lite single-pass reaches 0.896 exact F1 at ~$0.001/doc and 1.4 s median latency — far outpacing any agentic configuration on cost and speed while remaining competitive on accuracy.

- **Agentic multimodal benefits more from a tier upgrade** than single-pass. Claude’s agentic exact F1 jumps +11 pp from lite to standard; GPT’s improves +22 pp. Single-pass gains only 1–3 pp, pointing to a performance ceiling at lite that the heavier approach breaks through at standard.

- **`jurisdiction` and `party` are the most reliably extracted fields** across all experiments. `term` consistently lags by 5–25 pp, reflecting its higher linguistic variability and frequent absence in documents.

- **Per-document score variance is high** (std 0.15–0.29), indicating that document difficulty is a stronger driver of per-document outcomes than model choice for borderline cases.
